# Session 2 — Build an Agentic RAG System

In this session you will build, step by step, an agent that answers
questions about NVIDIA's financial results using the company's own filings.

The pipeline has four stages:

1. **Chunk** 
2. **Vectorize** 
3. **Search** 
4. **Assemble an agent** 

The corpus is in `data/02_processed/`: NVIDIA's 10-K filings for fiscal 2025 and 2026,
trimmed to their substantive pages, plus the transcripts of the two matching earnings
calls.

## Setup

We only need the standard library plus `pypdf` to read the filings. You should take a quick look before starting the exercise !

In [ ]:
import json
import re
from pathlib import Path
from tempfile import TemporaryDirectory

from pypdf import PdfReader

PROCESSED_DIR = Path("../data/02_processed")
CHUNKS_PATH = Path("../data/03_vectors/chunks.json")

sorted(p.name for p in PROCESSED_DIR.iterdir() if p.suffix in {".pdf", ".txt"})

## Exercise 1 — Chunking the documents

A model cannot read 157 pages at once, so we cut each document into chunks of a few
hundred characters.

- **Overlap** — consecutive chunks share their first and last few hundred characters, so
  a sentence cut by one boundary still appears whole in the next chunk.
  
- **Metadata** — document, page and year, so an answer can say where it came from.

Reading the file, joining the pages, tracking page positions, metadata and saving are
given. You write the **sliding window**: the three `TODO` lines.

In [ ]:
def chunk_text(input_path: str, output_path: str, chunk_size: int, overlap: int) -> None:
    """Split a document into overlapping chunks and save them with their metadata.

    Parameters
    ----------
    input_path : str
        Path to the document to chunk, either a `.pdf` filing or a `.txt` transcript.
    output_path : str
        Path to the JSON file the chunks are written to. Existing content is replaced.
    chunk_size : int
        Maximum number of characters in a chunk.
    overlap : int
        Number of characters each chunk shares with the previous one.

    Returns
    -------
    None
        The chunks are written to `output_path` as a list of dictionaries, each with
        the keys `id`, `text`, `document`, `page` and `year`.
    """
    path = Path(input_path)
    year = int(re.search(r"\d{4}", path.stem).group())

    if path.suffix == ".pdf":
        pages = [page.extract_text() or "" for page in PdfReader(path).pages]
    else:
        pages = [path.read_text()]

    text = "\n".join(pages)

    page_starts = []
    offset = 0
    for page in pages:
        page_starts.append(offset)
        offset += len(page) + 1

    chunks = []
    step = ...  # TODO 1 — how far apart should two consecutive chunks start?
    start = 0
    while start < len(text):
        piece = ...  # TODO 2 — the chunk_size characters beginning at `start`

        if piece.strip():
            page = sum(1 for page_start in page_starts if page_start <= start)
            chunks.append(
                {
                    "id": f"{path.stem}_p{page:03d}_c{len(chunks):04d}",
                    "text": piece,
                    "document": path.name,
                    "page": page,
                    "year": year,
                }
            )

        if start + chunk_size >= len(text):
            break
        start = ...  # TODO 3 — move on to where the next chunk begins

    Path(output_path).write_text(json.dumps(chunks, indent=2))

### Check your implementation

The test below chunks a document small enough to check by hand. With `chunk_size=4` and
`overlap=1`, consecutive chunks start 3 characters apart, so `"abcdefghij"` should come
out as `"abcd"`, `"defg"`, `"ghij"`.

In [ ]:
with TemporaryDirectory() as tmp:
    sample = Path(tmp) / "sample-2026.txt"
    sample.write_text("abcdefghij")
    output = Path(tmp) / "sample-chunks.json"
    chunk_text(str(sample), str(output), chunk_size=4, overlap=1)
    chunks = json.loads(output.read_text())

assert [c["text"] for c in chunks] == ["abcd", "defg", "ghij"], [c["text"] for c in chunks]
assert chunks[0]["document"] == "sample-2026.txt"
assert chunks[0]["year"] == 2026
print("OK —", len(chunks), "chunks")

### Chunk the real corpus

Now run it over the four documents. `chunk_size=1000` with `overlap=200` is a reasonable
starting point; you will get the chance to question both values later.

In [ ]:
CHUNKS_PATH.parent.mkdir(parents=True, exist_ok=True)
all_chunks = []

for document in sorted(PROCESSED_DIR.iterdir()):
    if document.suffix not in {".pdf", ".txt"}:
        continue
    per_document = CHUNKS_PATH.parent / f"{document.stem}-chunks.json"
    chunk_text(str(document), str(per_document), chunk_size=1000, overlap=200)
    all_chunks.extend(json.loads(per_document.read_text()))

CHUNKS_PATH.write_text(json.dumps(all_chunks, indent=2))
print(f"{len(all_chunks)} chunks written to {CHUNKS_PATH}")

## Appendix — going further

Cutting on a fixed number of characters is the simplest thing that works. What it costs,
visible in the chunks you just produced:

- **Tables lose their headers.** The 2026 income statement splits in two: the second chunk
  runs from `Gross profit` down to earnings per share with no header, so nothing says
  which figure belongs to which year. It does contain a header lower down, but that one
  belongs to the next statement. Fix: detect tables and never cut inside one.

- **A chunk can span two pages but records only the first**.

- **Chunks can begin mid-word.** 

- **One large JSON array** has to be loaded whole; `.jsonl` can be streamed.